# Description Experiments Notebook

This notebook documents the process of condensing and distilling the `racquet_description` column from a long-winded marketing brief to a concise and focused semantically-rich blurb.

## Data loading and imports

In [1]:
# All imports here
from pathlib import Path
import pprint
import pandas as pd

In [2]:
# Define data directory and load data
DATA_DIR = Path().cwd().parent / "data"
cleaned_data = pd.read_csv(DATA_DIR / "interim" / "racquets_cleaned_03_25_26.csv", index_col=0)
df = cleaned_data.copy()

## Racquet description overview

First, let's examine what the `racquet_description` column looks like. While there is some semantically useful information in there (e.g. "easy learning curve", "easy acceleration", "spin-friendly", etc), there is a lot of fluff too ("in a class by itself", "fabled combination", etc.)

In [3]:
pprint.pprint(df["racquet_description"][0])

('Engineered for speed. With its fabled combination of speed, power, and spin, '
 'the Pure Aero practically stands in a class by itself, not just for its '
 'legendary accomplishments, but also for its easy learning curve and '
 'universal appeal. Like the many versions that came before it, the Pure Aero '
 '2026 features the same basic spec formula that has attracted generations of '
 'players. With its sub-325 swingweight, it offers easy acceleration from the '
 'baseline and quick handling at net, delivering the kind of spin-friendly '
 'targeting that puts the lines in play. For 2026, Babolat reengineers the '
 'geometry of the shaft to reduce wind drag, giving this speedy racquet an '
 'even more explosive feel, making it even easier to whip up spin, generate '
 'pace, or reset the point with an outstretched flick. Other updates include '
 'the latest version of NF2 Tech, a dampening technology that uses flax fibers '
 'in strategic locations to provide a softer, more arm-friendl

I need to explore ways to strip out just the semantically useful information from the `racquet_description` columns. Since this is a relatively small dataset (only ~320 racquets), it's feasible to just pass them through an LLM to remove the marketing fluff.

## Experiments

In this section, I'll try different prompts (and potentially different models) to see what works best for creating a semantically richer `racquet_description` field. I'll start by creating a CSV with just the racquet IDs and descriptions of all racquets that are not null in `racquet_description`. The CSV is saved to the `data/interim` as `racquets_desc_llm_input_03_25_26.csv`.

In [4]:
# Write valid racquet ids and descriptions to a JSON file to pass to LLM
df_valid_desc = df.copy()
df_valid_desc = df_valid_desc[df_valid_desc["racquet_description"].isna() == False]
df_valid_desc[["racquet_id", "racquet_description"]].to_csv(DATA_DIR / "interim" / "racquets_desc_llm_input_03_25_26.csv")

### Experiment 1

For the first experiment, I asked ChatGPT to write me an optimized prompt (after describing what I needed it to do) and then used it. I examined its thinking while it was working through the CSV and found that: 

* It deemed 301 rows too many to manually read and process
* Decided to use heavy regex to try and parse out semantically important information
* Created programmatic fallbacks in case it didn't work properly

While this is not really what I wanted, I decided to examine the outputs anyways to see how well this approach fared. To see the prompt I used, reference `prompts/sem_description_v1.txt`. The outputted CSV file is stored at `data/interim/racquets_desc_llm_output_v1_03_25_26.csv`.

In [5]:
# Load in LLM output
llm_output_v1 = pd.read_csv(DATA_DIR / "interim" / "racquets_desc_llm_output_v1_03_25_26.csv").drop(columns = ["Unnamed: 0"])

In [6]:
llm_output_v1

,racquet_id,racquet_description,semantic_description
0,RCBAB-BPAR26,Engineered for speed. With its fabled combinat...,"This racquet offers added pop, easy access to ..."
1,RCBAB-BPA98R,Speed. Spin. Accuracy. Featuring the smallest ...,"This racquet offers controllable power, added ..."
2,RCBAB-BPA982,The most surgical member of the Pure Aero fami...,"This racquet offers controllable power, added ..."
3,RCBAB-BPAPLR,One of the games most powerful and spin-friend...,"This racquet offers easy power, easy access to..."
4,RCBAB-BPAT26,Updated with improved aerodynamics and better ...,"This racquet offers added pop, strong directio..."
...,...,...,...
296,RCSOLINCO-WO18XT,Solinco delivers a rare combination of plow-th...,This racquet offers strong directional control...
297,RCSOLINCO-SWHT28,"This 28"" racquet offers dangerous power on ser...",This racquet offers strong directional control...
298,RCSOLINCO-WHO30X,"This racquet's combination of power, accuracy ...",This racquet offers strong directional control...
299,RCLACTF-TRLA23,Elegant cosmetic. Easy Power. Spin-friendly Pr...,"This racquet offers easy power, strong directi..."


In [7]:
# Basic checks
assert len(df_valid_desc) == len(llm_output_v1)
assert set(df_valid_desc["racquet_id"]) == set(llm_output_v1["racquet_id"])
assert df_valid_desc["racquet_id"].is_unique
assert llm_output_v1["racquet_id"].is_unique

In [31]:
# Join to full df
df_merged_v1 = pd.merge(left = df, right = llm_output_v1[["racquet_id", "semantic_description"]], how = "left", 
         on = "racquet_id")

In [32]:
# Check that only len(df) - len(df_valid_desc) rows are NA -- FAILED
assert df_merged_v1["semantic_description"].isna().sum() == df.shape[0] - df_valid_desc.shape[0]

AssertionError: 

It looks like the LLM didn't parse all of the rows in the CSV I gave it. Now, I need to investigate exactly which rows were missed and what happened.

In [33]:
# Get IDs that are NA in merged df but not in original df
na_discrepancies = list(set(df_merged_v1[df_merged_v1["semantic_description"].isna()]["racquet_id"]) - set(df[df["racquet_description"].isna()]["racquet_id"]))
len(na_discrepancies)

3

In [34]:
# Display original description and NA value row side by side
df_merged_v1[df_merged_v1["racquet_id"].isin(na_discrepancies)][["racquet_id", "racquet_description", "semantic_description"]]

,racquet_id,racquet_description,semantic_description
96,RCHEAD-HSPELT,Head serves up the perfect racquet for value h...,NaN
99,RCHEAD-HSPDPL,Head celebrates one of the game's most legenda...,NaN
101,RCHEAD-HSPDP,Updated with Auxetic 2 for 2024,NaN


In [35]:
# Check that failed columns were in df_valid_desc
df_valid_desc[df_valid_desc["racquet_id"].isin(na_discrepancies)]

,racquet_id,racquet_url,racquet_name,racquet_brand,racquet_img,racquet_rating,racquet_price,racquet_description,racquet_swingweight,racquet_composition,...,racquet_length_in,racquet_strung_weight_oz,racquet_balance_in,racquet_balance_HH_HL,racquet_stiffness,racquet_avg_beam_width,racquet_mains,racquet_crosses,racquet_tension_lower,racquet_tension_upper
122,RCHEAD-HSPELT,https://www.tennis-warehouse.com/Head_Speed_El...,Head Speed Elite 2026,Head,https://img.tennis-warehouse.com/watermark/rs....,NaN,149.0,Head serves up the perfect racquet for value h...,303.0,Auxetic 2/Graphene Inside/Boron/Graphite,...,27.0,10.0,13.19,2.0,63.0,25.0,16.0,19.0,48.0,57.0
126,RCHEAD-HSPDPL,https://www.tennis-warehouse.com/Head_Speed_Pr...,Head Speed Pro Legend,Head,https://img.tennis-warehouse.com/watermark/rs....,4.5,199.0,Head celebrates one of the game's most legenda...,333.0,Auxetic 2/Graphene Inside/Graphite,...,27.0,11.0,12.80,6.0,60.0,23.0,18.0,20.0,48.0,57.0
128,RCHEAD-HSPDP,https://www.tennis-warehouse.com/Head_Speed_Pr...,Head Speed Pro,Head,https://img.tennis-warehouse.com/watermark/rs....,4.5,199.0,Updated with Auxetic 2 for 2024,333.0,Auxetic 2/Graphene Inside/Graphite,...,27.0,11.0,12.80,6.0,60.0,23.0,18.0,20.0,48.0,57.0


In [36]:
df_merged_v1[df_merged_v1["racquet_id"].isin(na_discrepancies)]["racquet_description"].iloc[0]

'Head serves up the perfect racquet for value hunters'

In [37]:
df_merged_v1[df_merged_v1["racquet_id"].isin(na_discrepancies)]["racquet_description"].iloc[1]

"Head celebrates one of the game's most legendary racquets with a sleek black cosmetic"

In [38]:
df_merged_v1[df_merged_v1["racquet_id"].isin(na_discrepancies)]["racquet_description"].iloc[2]

'Updated with Auxetic 2 for 2024'

Based on the results above, it looks like Chat GPT simply skipped these 3 rows because they didn't contain any semantically useful information. All in all, this isn't too big of a deal as it's trivial to replace these NAs with empty strings along with the other valid NAs.

However, this is a significant red mark when considering this approach's validity in a production environment. Before moving forward, I'm going to try a different prompt that more explictly instructs ChatGPT to use its language model on each row and to make sure that each row returns a non-null description (should fill with "" if no useful information).

In [48]:
df_merged_v1["semantic_description"] = df_merged_v1["semantic_description"].fillna("")

### Experiment 2

For the second experiment, I asked ChatGPT to adjust the prompt to ensure that it actually read each description and to only use Python for file I/O and row tracking. I also added a clause specifically instructing it to use empty quotes if it can't find anything to extract. I examined its thinking while it was working through the CSV and found that: 

* It, again, deemed 301 rows too many to manually read and process
* Stopped early

This time ChatGPT flat out refused to completely finish the task as it recognized that it couldn't complete it to the specifications I requested. I decided to ask Gemini if it could do the task and it said yes. So, I used the same prompt and examined its thinking while it was working through the CSV. I found that:

* It seemed to understand the need for internal chunking like I asked
* It appeared to follow instructions through row 50 but then suddenly terminated and said it was having trouble
* After being prompted to think about why it was having trouble and to try again, it explicitly decided to read the CSV in chunks, write the output to a temp file and continue until it got through all 301 racquets
* It explicitly confirmed that it needed to use its language capabilities for the rewrites 
* It made it slightly after (around 200) before crashing again.